# LangChain 概览

本 Notebook 从全局视角介绍 LangChain 是什么、解决了什么问题、核心组件如何协作。

## 什么是 LangChain？

LangChain 是一个**构建 LLM 应用的框架**，解决三个核心问题：

1. **连接** — 把 LLM 和外部数据（文档、数据库、API）连接起来
2. **编排** — 把多个 LLM 调用组合成复杂的工作流
3. **智能体** — 让 LLM 自主决定调用哪些工具、按什么顺序调用

## 为什么需要 LangChain？

直接用 LLM API（如 DeepSeek、ChatGPT）只能做**一问一答**。实际应用需要：

| 场景 | 直接调 API | LangChain |
|------|-----------|-----------|
| 多轮对话 | 自己管理消息历史 | Messages 体系自动管理 |
| 使用工具/API | 自己写解析循环 | create_agent 自动处理 |
| 读取文档 | 自己实现分割、向量化 | Document Loaders + VectorStore |
| 复杂流程 | 硬编码 if/else | LCEL / LangGraph 可视化编排 |

LangChain 提供**标准化的组件和接口**，让你专注于业务逻辑，而不是重复造轮子。

In [ ]:
# 验证环境
import langchain
from langchain_learning.llm import get_llm

print(f"LangChain 版本: {langchain.__version__}")

llm = get_llm()
r = llm.invoke("用一句话说明 LangChain 是什么")
print(f"LLM 调用正常: {r.content}...")

## 整体架构

LangChain 1.0 以 `create_agent` 为核心构建，底层基于 LangGraph 运行时。

> 参考：[LangChain 1.0 发布博客](https://www.langchain.com/blog/langchain-langgraph-1dot0)

### 核心 Agent 循环

```mermaid
%%{init: {'theme': 'dark'}}%%
graph LR
    subgraph Setup
        A[选择模型] --> B[绑定工具]
        B --> C[设置提示词]
    end

    subgraph Execution
        C --> D[用户输入]
        D --> E{LLM 判断}
        E -->|调用工具| F[执行工具]
        F --> D
        E -->|直接回答| G[最终输出]
    end
```

### 三层关系

```mermaid
%%{init: {'theme': 'dark'}}%%
graph TD
    H[LangChain<br/>create_agent<br/>高层 API] --> L[LangGraph<br/>底层运行时]
    L --> S[StateGraph<br/>状态管理]
    L --> P[Persistence<br/>持久化]
    L --> HIL[Human-in-the-Loop<br/>人工介入]
```

**术语：**

### 核心层（Core Components）

| 组件 | 作用 | 类比 |
|------|------|------|
| **Chat Models** | 调用 LLM（DeepSeek、GPT 等） | "引擎" |
| **Messages** | System / Human / AI 消息类型 | "对话记录" |
| **Prompt Templates** | 带变量的提示词模板 | "填空模板" |
| **Output Parsers** | 把 LLM 输出转为结构化数据 | "翻译器" |
| **Tools** | 给 LLM 调用的函数（查天气、计算等） | "工具箱" |
| **Document Loaders** | 加载 PDF/网页/数据库 | "读卡器" |
| **Vector Stores** | 向量数据库（ChromaDB、FAISS） | "记忆库" |
| **Embeddings** | 把文本转为向量 | "指纹提取" |

### 编排层（Orchestration）

| 组件 | 作用 |
|------|------|
| **LCEL** | 用 `\|` 把组件串成链，声明式语法 |
| **LangGraph** | 用图定义复杂工作流（状态机） |

### 应用层

| 组件 | 作用 |
|------|------|
| **Agent** | 能自主使用工具的智能体 |
| **RAG** | 检索+生成，基于文档回答问题 |

### 基础设施

| 组件 | 作用 |
|------|------|
| **Callbacks** | 监听执行过程（日志、耗时统计） |
| **LangSmith** | 链路追踪、调试、评估 |

## 核心数据流：Chain（链）

从用户输入到输出的典型流程：

```mermaid
%%{init: {'theme': 'dark'}}%%
sequenceDiagram
    participant User
    participant Prompt as PromptTemplate
    participant LLM
    participant Parser as OutputParser
    
    User->>Prompt: 输入 + 变量
    Prompt->>LLM: 格式化后的提示词
    LLM->>Parser: 原始文本输出
    Parser->>User: 结构化结果
```

LCEL 中就是：`prompt | llm | parser`。

更复杂的 Agent 场景：

```mermaid
%%{init: {'theme': 'dark'}}%%
sequenceDiagram
    participant User
    participant Agent
    participant Tool
    participant LLM
    
    User->>Agent: 提问
    Agent->>LLM: 理解问题
    LLM-->>Agent: 决定调工具
    Agent->>Tool: 调用工具
    Tool-->>Agent: 返回结果
    Agent->>LLM: 分析结果
    LLM-->>Agent: 生成回答
    Agent->>User: 最终回复
```

## 学习路径

```mermaid
%%{init: {'theme': 'dark'}}%%
timeline
    title LangChain 学习路线
    第一阶段 核心组件 : Chat Models : Prompt / Parser : LCEL : Tools : Streaming : Agent
    第二阶段 RAG : Document Loaders : Embeddings : VectorStore : RAG Chain
    第三阶段 LangGraph : StateGraph : Subgraphs : Persistence : Streaming
    第四阶段 Agent 系统 : Tool Calling : Multi-agent : Human-in-the-Loop
    第五阶段 生产部署 : Testing : Deploy : Agentic RAG 实战
```

## 本项目配置

- **模型**：DeepSeek（OpenAI 兼容 API）
- **SDK**：langchain-openai（ChatOpenAI）
- **工具统一入口**：`get_llm()` — 在 `src/langchain_learning/llm.py` 中
- **包管理**：uv，依赖见 `pyproject.toml`
- **代码规范**：ruff

In [ ]:
from langchain_learning.llm import get_llm

# 默认 DeepSeek V3
llm = get_llm()

# 或切换为推理模型
reasoner = get_llm(model="deepseek-reasoner")

## 接下来的 Notebook

| # | Notebook | 学完你会 |
|---|----------|----------|
| 01 | Chat Models | 调用 LLM、多消息对话、流式输出 |
| 02 | Prompt / Parser | 模板变量、结构化输出 |
| 03 | LCEL | 用 `\|` 构建链、并行执行 |
| 04 | Tools | 定义工具、绑定 LLM |
| 05 | Streaming | 流式、事件流、回调 |
| 06 | Middleware | 日志、耗时统计、自定中间件 |
| 07 | Agent | create_agent 智能体 |